# MaxEnt experiment notebook

Interactive one-model MaxEnt runs. Steps mirror `sdm train` using the same public library functions — no extra façade.

## How to run

1. From the repo root: `uv sync`, then open this notebook.
2. Set **Model** (`LATIN_NAME`, `ACTIVITY_TYPE`). Default: `Plecotus auritus` / `Roost`.
3. Run **Configs + load data**, then **Prepare once** (background draw + elapid annotate — slow; keep results in kernel memory).
4. Edit **Overrides** (variables / MaxEnt / background prep). Leave empty to match committed config.
5. Run **Species prep + train** (`prepare_species_training_data` → `adapter.train`). Re-run after param edits without re-annotating unless you change surface background knobs (then re-run Prepare).
6. Optional: **Full reproduce** via `train_sdm_models` (same entry as `sdm train`).
7. Inspection cells write under a local temp dir only — **never** the prediction publish bucket.

**Public / privacy:** clear all cell outputs before commit. Do not embed record points, site/roost names, grid references, or secrets.


## Setup


In [ ]:
from copy import deepcopy
from dataclasses import replace
from pathlib import Path
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sdm.commands.modelling.explain_sdm_models import (
    compute_shap_for_model,
    plot_filtered_shap_dependence,
)
from sdm.commands.modelling.train_sdm_models import (
    prepare_species_training_data,
    prepare_training_data,
    simplify_boundary,
    train_sdm_models,
)
from sdm.commands.modelling.utils import get_model_id
from sdm.data.loaders.vector import load_bat_data
from sdm.models.adapters import get_adapter
from sdm.models.maxent.folds import resolve_fold_params
from sdm.models.maxent.maxent_model import DefaultMaxentConfig, apply_models_to_raster
from sdm.models.maxent.training import summarize_cv_scores
from sdm.occurrence.background_settings import resolve_background_settings
from sdm.raster.io import load_environmental_variables
from sdm.types import TrainingData
from sdm.utils.fs import local_materialized_path
from sdm.utils.io import (
    CONFIG_PATH,
    MODEL_CONFIG_PATH,
    load_boundary,
    load_model_config,
    load_project_config,
    load_variables_config,
    set_project_wd,
)
from sdm.utils.model_runs import resolve_data_path
from sdm.utils.sdm_config import (
    model_config_file_paths,
    resolve_model_config_dir,
    resolve_sdm_config_root,
)
from sdm.viz.plots import generate_pdp_individual_plots
from sdm.viz.prediction_maps import plot_prediction_map

set_project_wd()
SCRATCH = Path(tempfile.gettempdir()) / "sdm-maxent-experiment"
SCRATCH.mkdir(parents=True, exist_ok=True)
print("cwd ready; scratch:", SCRATCH)


## Model

Committed per-model YAML: `data/sdm_config/<id>/`. Project `config.yml` + `model_config.yml` supply seed (42), 10 km spatial-block CV × 5, and background defaults.


In [ ]:
LATIN_NAME = "Plecotus auritus"
ACTIVITY_TYPE = "Roost"
CONFIG_DIR = Path("data/sdm_config")
OUTPUT_DIR = SCRATCH / "runs"  # local only — never prediction_publish
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional path overrides (None → config.yml).
BATS_FILE = None
EV_FILE = None

print(f"Model: {LATIN_NAME} / {ACTIVITY_TYPE}")
print(f"Local output: {OUTPUT_DIR}")


## Configs + load data


In [ ]:
project_config = load_project_config(CONFIG_PATH)
base_model_cfg = load_model_config(resolve_sdm_config_root(MODEL_CONFIG_PATH))
training_seed = base_model_cfg.seed

bg = resolve_background_settings(
    project_config=project_config,
    model_config=base_model_cfg,
    seed=training_seed,
)
fold_params = resolve_fold_params(base_model_cfg.cv, model_seed=training_seed)

bats_uri = resolve_data_path(BATS_FILE or project_config.paths.occurence_data)
ev_uri = resolve_data_path(EV_FILE or project_config.paths.ev_tiff)
boundary_uri = resolve_data_path(project_config.paths.boundary)

with local_materialized_path(bats_uri) as bats_path:
    bats = load_bat_data(bats_path)
bats = bats[bats.accuracy <= 100]
all_bats = bats  # background density uses every bat (same as sdm train)
filtered = bats[
    (bats.latin_name == LATIN_NAME) & (bats.activity_type == ACTIVITY_TYPE)
].copy()
assert not filtered.empty, f"No rows for {LATIN_NAME!r} / {ACTIVITY_TYPE!r}"

with local_materialized_path(boundary_uri) as boundary_path:
    boundary = load_boundary(filepath=boundary_path, buffer_distance=0, target_crs=27700)
boundary = simplify_boundary(boundary, tolerance=100)

evs_to_model, _ = load_environmental_variables(ev_uri)

model_id = get_model_id([LATIN_NAME, ACTIVITY_TYPE])
model_subdir = resolve_model_config_dir(CONFIG_DIR, model_id, must_exist=True)
per_model_cfg_path, per_model_vars_path = model_config_file_paths(model_subdir)
per_model_cfg = load_model_config(per_model_cfg_path, fallback_seed=training_seed)
variables_cfg = load_variables_config(per_model_vars_path)
committed_features = list(variables_cfg.variables)
maxent_config = DefaultMaxentConfig.from_config(per_model_cfg)
maxent_config.class_weights = bg.maxent_class_weights()

print("Seed:", training_seed)
print("CV:", fold_params.to_metadata())
print("Background:", bg.to_metadata())
print(f"Features ({len(committed_features)}):", committed_features)


## Prepare once (cache in kernel)

`prepare_training_data`: shared background draw + elapid annotate. Re-run only when switching model or changing **surface** background knobs (`n_background_points`, `surface`, `seed`, method/sigma, …).


In [ ]:
presence_with_evs, background_with_evs, ev_columns = prepare_training_data(
    occurrence_gdf=filtered,
    background_occurrence_gdf=all_bats,
    boundary=boundary,
    evs_to_model=evs_to_model,
    n_background_points=bg.n_background_points,
    background_method=bg.background_method,
    background_value=bg.background_value,
    sigma=bg.sigma,
    transform_method=bg.transform_method,
    seed=bg.seed,
    surface=bg.surface,
    grid_resolution=bg.surface_resolution_m,
)
print(
    f"Annotated: {len(presence_with_evs)} presence, "
    f"{len(background_with_evs)} background; {len(ev_columns)} EV columns"
)
baseline_mean_cv = None  # set after first successful train below
baseline_std_cv = None


## Overrides (edit here)

Leave as committed defaults for reproduction. After edits, re-run **Species prep + train**. If you change surface BG knobs, also re-run **Prepare once** (and update `bg` via `replace` or re-resolve).


In [ ]:
# --- edit me ---
VARIABLES = None  # e.g. committed_features[:8]; None → committed_features
MAXENT_OVERRIDES = {
    # "beta_multiplier": 3.0,
    # "feature_types": ["linear", "hinge"],
}
# Prep-only knobs (no annotate redraw). Surface knobs → re-run Prepare with updated bg.
BG_PREP_OVERRIDES = {
    # "presence_thin_m": 1000.0,
    # "presence_exclusion_m": 250.0,
}
# --- end edits ---

features = list(VARIABLES) if VARIABLES is not None else list(committed_features)
missing = set(features) - set(ev_columns)
assert not missing, f"Variables not in EV stack: {sorted(missing)}"

bg_train = replace(bg, **BG_PREP_OVERRIDES) if BG_PREP_OVERRIDES else bg
maxent_train = deepcopy(maxent_config)
for key, value in MAXENT_OVERRIDES.items():
    setattr(maxent_train, key, value)
maxent_train.class_weights = bg_train.maxent_class_weights()

print("Train features:", features)
print("MaxEnt overrides:", MAXENT_OVERRIDES or "(none)")
print("BG prep overrides:", BG_PREP_OVERRIDES or "(none)")


## Species prep + train

Same modular path as `train_sdm_models` after shared annotate: `prepare_species_training_data` → `get_adapter(...).train`.


In [ ]:
t0 = time.perf_counter()
species_data = prepare_species_training_data(
    presence_data=presence_with_evs,
    background_data=background_with_evs,
    latin_name=LATIN_NAME,
    activity_type=ACTIVITY_TYPE,
    ev_columns=ev_columns,
    grid_size_m=bg_train.presence_thin_m,
    d_min=bg_train.presence_exclusion_m,
    d_max=bg_train.presence_exclusion_max_m,
    sample_weight_n_neighbors=bg_train.sample_weight_n_neighbors,
    random_state=bg_train.seed,
    presence_thin_m=bg_train.presence_thin_m,
    background_thin_m=bg_train.background_thin_m,
    distance_weights=bg_train.distance_weights,
    background_distance_weights=bg_train.background_distance_weights,
    cap_ratio=bg_train.cap_ratio,
    min_background=bg_train.min_background,
    max_background=bg_train.max_background,
)
assert not species_data.empty and "class" in species_data.columns

training = [
    TrainingData(
        latin_name=LATIN_NAME,
        activity_type=ACTIVITY_TYPE,
        occurrence=species_data,
        maxent_config=maxent_train,
        model_features=features,
        model_limited=bool(per_model_cfg.model_limited),
    )
]
adapter = get_adapter(base_model_cfg.type)
bundle = adapter.train(
    training,
    seed=training_seed,
    fold_params=fold_params,
    n_jobs=1,
    max_threads_per_model=1,
)
result = bundle.results[0]
assert result.success, result.error
mean_cv, std_cv, n_valid, n_total = summarize_cv_scores(result.cv_scores)
elapsed = time.perf_counter() - t0

if baseline_mean_cv is None:
    baseline_mean_cv, baseline_std_cv = mean_cv, std_cv
    print(f"CV AUC: {mean_cv:.4f} ± {std_cv:.4f} ({n_valid}/{n_total} folds) — set as baseline")
else:
    delta = mean_cv - baseline_mean_cv
    print(
        f"CV AUC: {mean_cv:.4f} ± {std_cv:.4f} "
        f"(baseline {baseline_mean_cv:.4f} ± {baseline_std_cv:.4f}; Δ {delta:+.4f})"
    )
print(f"Features ({len(features)}): {features}")
print(f"Retrain wall-clock: {elapsed:.1f}s")
final_model = result.final_model


## Optional: full reproduce via `train_sdm_models`

Same entry point as `sdm train`. Use when you want package/threshold writes without stepping through cells. Skip for day-to-day param tweaks (Prepare + train above is faster once annotate is cached).


In [ ]:
# Uncomment to run a full one-model CLI-equivalent train into local scratch.
# repro_df = train_sdm_models(
#     species=[LATIN_NAME],
#     activity_types=[ACTIVITY_TYPE],
#     config_dir=CONFIG_DIR,
#     bats_file=BATS_FILE,
#     ev_file=EV_FILE,
#     output_dir=OUTPUT_DIR / "cli-repro",
#     n_jobs=1,
#     verbose=True,
# )
# print(repro_df[["identifier", "mean_cv_score", "std_cv_score"]].to_string(index=False))


## Inspection — response curves (PDP)

Feature space only; plots under local scratch.


In [ ]:
training_gdf = training[0].occurrence
feature_frame = training_gdf[features].dropna()
sample = feature_frame.sample(n=min(500, len(feature_frame)), random_state=training_seed)
models_df = pd.DataFrame(
    [
        {
            "model_object": final_model,
            "species": LATIN_NAME,
            "activity": ACTIVITY_TYPE,
            "feature_names_used": features,
        }
    ]
)
pdp_dir = Path(tempfile.mkdtemp(prefix="maxent-pdp-", dir=SCRATCH))
pdp_results = generate_pdp_individual_plots(
    models_df=models_df,
    ev_sample=sample,
    output_dir=pdp_dir,
    grid_resolution=30,
    n_workers=1,
)
print("PDP plots:", pdp_dir)
pdp_results.head()


## Inspection — SHAP importance / dependence

No record coordinates — feature space only.


In [ ]:
explainer, shap_values, X_explain, _background = compute_shap_for_model(
    final_model,
    feature_frame,
    features,
    n_background=min(100, len(feature_frame)),
    n_explain=min(200, len(feature_frame)),
    random_state=training_seed,
)
shap.plots.bar(shap_values, show=True)
mean_abs = np.abs(shap_values.values).mean(axis=0)
top = features[int(np.argmax(mean_abs))]
fig, ax = plot_filtered_shap_dependence(top, X_explain, shap_values)
plt.show()


## Inspection — quick prediction map

Local temp GeoTIFF only. Do not point at `prediction_publish.root` / the bucket.


In [ ]:
pred_dir = Path(tempfile.mkdtemp(prefix="maxent-pred-", dir=SCRATCH))
pred_path = pred_dir / "suitability.tif"
plot_path = pred_dir / "suitability_preview.png"
model_key = get_model_id([LATIN_NAME, ACTIVITY_TYPE])

apply_models_to_raster(
    models={model_key: final_model},
    raster_path=ev_uri,
    output_path=pred_path,
    predict_proba=True,
    quiet=True,
)
print("Local prediction (not published):", pred_path)

plot_prediction_map(
    pred_path,
    output_path=plot_path,
    title=f"{LATIN_NAME} / {ACTIVITY_TYPE}",
)
print("Preview PNG:", plot_path)
